# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment.


In [ ]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token**, as in the data contract: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, `scipy` supplies the negative binomial luck test behind real shortfalls and rule B, `scikit-learn` supplies logistic regression for the confidence label, `matplotlib` draws the one chart, and `json` writes the summary for the paper.
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which saves each result once under `work/outputs/w03_cache/` and reuses it on later runs.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [ ]:
import getpass
import os
import sys
import tempfile
import urllib.request

import duckdb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager
from matplotlib.lines import Line2D
from scipy.stats import nbinom
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## Data

The queue uses the same rules, target, spread factor, and 25 features as the modelling and validation notebooks. June 2026 was opened once by the validation audit; it is read again here as an ordinary month, both as the outcome month of the back-test and as the month the live queue is built from.


**Method:** this cell loads the page-month table for February to May 2026 with the same query and cache as the earlier notebooks, each client's search-history start date, and the content table's type, keyword-research fields, and latest optimization date.


In [ ]:
# The same page-month query and cache as the earlier notebooks.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"SELECT content_hash_id, content_type, main_intent, search_volume, competition, cpc, keyword_token_count, last_optimized_date FROM {DIM_CONTENT}").df()
show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with content type, keyword research fields, and latest optimization date"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Method:** this cell applies the data contract's rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair). For each month it computes the impression-weighted position, its daily spread, expected clicks from the fine position bins and from the five tiers (both client-scaled), and whether the page fell short by the locked rule: clicks at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target. The month M+1 version of the rule is the target.


In [ ]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the earlier score: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def add_month_measures(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add position, its spread, click-through, and both expectations for one month."""
    impressions, clicks = rows[f"impressions_{suffix}"], rows[f"clicks_{suffix}"]
    rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
    rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows[f"ctr_{suffix}"] = clicks / impressions
    rows[f"expected_bins_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_bin(rows[f"position_{suffix}"]))
    rows[f"expected_tier_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_tier(rows[f"position_{suffix}"]))
    return rows


def add_luck_test(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add the negative binomial luck test and the locked shortfall rule for one month."""
    clicks, expected = rows[f"clicks_{suffix}"], rows[f"expected_bins_{suffix}"]
    rows[f"p_{suffix}"] = nbinom.cdf(clicks, 1 / ALPHA, 1 / (1 + ALPHA * expected))
    rows[f"shortfall_{suffix}"] = (rows[f"p_{suffix}"] < 0.05) & (clicks <= 0.5 * expected)
    return rows


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with both months' measures and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    rows = add_month_measures(add_month_measures(rows, "m"), "next")
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    add_luck_test(add_luck_test(rows, "m"), "next")
    rows["target"] = rows["shortfall_next"]
    rows["missed_next"] = (rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

**Method:** this cell builds the 25 features with the modelling notebook's own code.


In [ ]:
# The leakage check's 23 features, plus rule B's two month-M signals.
CONTENT_TYPES = sorted(content["content_type"].dropna().unique())
INTENTS = sorted(content["main_intent"].dropna().unique())
PRODUCT_FIELDS = {"last_optimized_date", "optimization_eligible_date", "is_deleted", "is_published", "content_updated_date"}


def feature_vector(rows: pd.DataFrame) -> pd.DataFrame:
    """Return the model-ready feature matrix for one pair: month-M search features and keyword-research features only."""
    features = pd.DataFrame(index=rows.index)
    features["log_impressions_m"] = np.log1p(rows["impressions_m"])
    features["log_clicks_m"] = np.log1p(rows["clicks_m"])
    features["ctr_m"] = rows["clicks_m"] / rows["impressions_m"]
    features["position_m"] = rows["position_m"]
    features["position_volatility_m"] = rows["position_volatility_m"]
    features["days_with_impressions_m"] = rows["days_with_impressions_m"]
    features["log_expected_m"] = np.log1p(rows["expected_tier_m"])
    for field in ("search_volume", "competition", "cpc", "main_intent"):
        features[f"has_{field}"] = rows[field].notna().astype(int)
    features["log_search_volume"] = np.log1p(rows["search_volume"]).fillna(0)
    features["competition"] = rows["competition"].fillna(0)
    features["log_cpc"] = np.log1p(rows["cpc"]).fillna(0)
    features["keyword_token_count"] = rows["keyword_token_count"]
    for value in CONTENT_TYPES:
        features[f"content_type={value}"] = (rows["content_type"] == value).astype(int)
    for value in INTENTS:
        features[f"intent={value}"] = (rows["main_intent"] == value).astype(int)
    features["intent=unknown"] = rows["main_intent"].isna().astype(int)
    features["log_missed_m"] = np.log1p((rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0))
    features["short_m"] = rows["shortfall_m"].astype(int)
    return features


vectors = {name: feature_vector(rows) for name, rows in pairs.items()}
FEATURES = list(next(iter(vectors.values())).columns)
for name, features in vectors.items():
    assert list(features.columns) == FEATURES, f"{name}: columns differ from the first pair."
    assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes), f"{name}: a feature is not numeric."
    assert not features.isna().any().any(), f"{name}: a feature still has gaps."
assert not [column for column in FEATURES if column.endswith("_next")], "A month M+1 field reached the features."
assert not set(FEATURES) & PRODUCT_FIELDS, "A product field reached the features."

LABELS = {"log_impressions_m": "Impressions (log)", "log_clicks_m": "Clicks (log)", "ctr_m": "Click-through rate",
          "position_m": "Position", "position_volatility_m": "Position spread", "days_with_impressions_m": "Days with impressions",
          "log_expected_m": "Expected clicks (log)", "has_search_volume": "Has search volume", "has_competition": "Has competition",
          "has_cpc": "Has CPC", "has_main_intent": "Has search intent", "log_search_volume": "Search volume (log)",
          "competition": "Competition", "log_cpc": "CPC (log)", "keyword_token_count": "Keyword length (words)",
          "log_missed_m": "Missed clicks (log)", "short_m": "Short by the target's rule"}


def label(feature: str) -> str:
    """Return a plain-language name for a feature."""
    return LABELS.get(feature) or feature.replace("content_type=", "Content type: ").replace("intent=", "Search intent: ")
show_table("Feature vector, every pair", pd.DataFrame({
    "Rows": [f"{len(features):,}" for features in vectors.values()],
    "Features": [features.shape[1] for features in vectors.values()],
    "Gaps left": [int(features.isna().sum().sum()) for features in vectors.values()],
}, index=pd.Index(list(vectors), name="Pair")))

**Method:** this cell adds June 2026, builds the May→June pair with the data contract's rules and the spread factor estimated on the development pairs, and lists the four labelled pairs available for training.


In [ ]:
BACKTEST_MONTH, LIVE_MONTH = "2026-05", "2026-06"
page_months = pd.concat([page_months, cached(f"page_month_v3_{LIVE_MONTH}", PAGE_MONTH_SQL.format(month=LIVE_MONTH, source=month_files([LIVE_MONTH])))], ignore_index=True)
outcome_name = f"{BACKTEST_MONTH} → {LIVE_MONTH}"
outcome_pair = build_pair(BACKTEST_MONTH, LIVE_MONTH)
add_luck_test(add_luck_test(outcome_pair, "m"), "next")  # the spread factor stays the one estimated on the development pairs
outcome_pair["target"] = outcome_pair["shortfall_next"]
outcome_pair["missed_next"] = (outcome_pair["expected_bins_next"] - outcome_pair["clicks_next"]).clip(lower=0).where(outcome_pair["target"], 0)
labelled = {**pairs, outcome_name: outcome_pair}
labelled_vectors = {name: feature_vector(rows) for name, rows in labelled.items()}
show_table(f"Labelled month pairs (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in labelled.values()],
    "Real shortfalls": [f"{int(rows['target'].sum()):,}" for rows in labelled.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in labelled.values()],
    "Used to train": ["back-test and live model" if name in pairs else "live model only" for name in labelled],
}, index=pd.Index(list(labelled), name="Pair")))

## 1. Ranked actions + reason codes

**How the queue is built, fixed before any result in this notebook:**

1. **Who is scored:** every page visible in the month with a valid position, on a client whose search history covers the whole month, from that month's data alone. Pages the data contract would later drop stay in, because nobody knows on the first of the month which ones FlyRank will optimize or which will vanish.
2. **Who enters the queue:** pages with a real shortfall this month (at most half of expected clicks, negative binomial p below 0.05).
3. **Order:** rule B, the clicks missed against the position-bin expectation. The validation audit's sealed test kept rule B as the frozen ranking, and its top 50 held more of next month's missed clicks in every test set.
4. **Confidence:** logistic regression's probability that the shortfall is real again next month: high at 0.60 or more, medium from 0.30, low below 0.30. Logistic regression without class weights gives roughly honest probabilities, so the cut-offs read as plain chances.
5. **Reason codes and one action per page,** in this priority order:

| Reason code           | When it applies (month M only)                                   | Action                              |
| --------------------- | ---------------------------------------------------------------- | ----------------------------------- |
| `visibility_falling`  | Impressions below half of the month before                       | Check indexing and rankings first   |
| `rank_unstable`       | Daily position spread of 5 places or more                        | Wait and re-check next month        |
| `client_wide_low_ctr` | The whole site earns under half the clicks its positions predict | Review site-wide                    |
| `commercial_intent`   | Commercial or transactional keyword                              | Check the search results page first |
| `low_engagement`      | 30 or more GA4 sessions, under half of them engaged              | Improve the content                 |
| `real_shortfall` only | None of the above                                                | Rewrite the title and description   |

`visibility_falling` is new: every miss of the model in the modelling notebook was a page that lost its search visibility the following month, so a page already sliding out of search goes to an indexing check, not a title rewrite.


### 1a. The back-test: May's queue against June

**Method:** this cell defines the queue, scores every page visible in May 2026 with the model trained on the three development pairs, builds May's queue, and records what happened to each queued page in June.


In [ ]:
# Every threshold named once.
UNSTABLE_SPREAD, MIN_SESSIONS, ENGAGED_SHARE, CLIENT_LOW, VISIBILITY_DROP = 5, 30, 0.5, 0.5, 0.5
HIGH, MEDIUM = 0.60, 0.30
QUEUE_SIZE, CLIENT_CAP, LOST_SHARE = 50, 10, 0.2
BUYING = {"commercial", "transactional"}
MONTH_COLUMNS = ["impressions", "clicks", "position_impressions", "sum_position", "sum_position_sq", "clicks_without_position",
                 "days_with_impressions", "ga4_sessions", "ga4_engaged_sessions"]
ACTION_ORDER = [("visibility_falling", "Check indexing and rankings first"), ("rank_unstable", "Wait and re-check next month"),
                ("client_wide_low_ctr", "Review site-wide"), ("commercial_intent", "Check the search results page first"),
                ("low_engagement", "Improve the content")]
DEFAULT_ACTION = "Rewrite the title and description"


def rule_b(rows: pd.DataFrame) -> pd.Series:
    """The frozen ranking: missed clicks against the position-bin expectation, kept where the page fell short in month M."""
    return (rows["expected_bins_m"] - rows["clicks_m"]).clip(lower=0).where(rows["shortfall_m"], 0)


def client_factor(rows: pd.DataFrame) -> pd.Series:
    """Return each page's client factor: the client's clicks ÷ its expected clicks from position bins alone, +1 on both sides."""
    group = position_bin(rows["position_m"])
    raw = rows["impressions_m"] * group.map(rows["clicks_m"].groupby(group).sum() / rows["impressions_m"].groupby(group).sum())
    by_client = rows.assign(_raw=raw).groupby("client_hash_id")
    return rows["client_hash_id"].map((by_client["clicks_m"].sum() + 1) / (by_client["_raw"].sum() + 1))


def fit_logistic(names: list[str]) -> Pipeline:
    """Fit logistic regression, with the modelling notebook's settings, on the listed labelled pairs."""
    target = pd.concat([labelled[name]["target"] for name in names], ignore_index=True).astype(int)
    features = pd.concat([labelled_vectors[name] for name in names], ignore_index=True)
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2_000)).fit(features, target)


def month_rows(month: str, model: Pipeline) -> pd.DataFrame:
    """Score every page visible in one month from that month's data alone: rule B, the model's probability, reason codes, and an action."""
    rows = page_months[(page_months["month"] == month) & (page_months["position_impressions"] > 0)]
    rows = rows[rows["client_hash_id"].isin(covered_clients(month))]
    rows = rows.rename(columns={column: f"{column}_m" for column in MONTH_COLUMNS}).merge(content, on="content_hash_id", how="left")
    rows = add_luck_test(add_month_measures(rows.reset_index(drop=True), "m"), "m")
    previous = (pd.Period(month, freq="M") - 1).strftime("%Y-%m")
    rows["impressions_prev"] = rows["content_hash_id"].map(page_months[page_months["month"] == previous].set_index("content_hash_id")["impressions"])
    assert not [column for column in rows.columns if column.endswith("_next")], "A next-month column reached the scoring."
    rows["rule_b"] = rule_b(rows)
    rows["probability"] = model.predict_proba(feature_vector(rows))[:, 1]
    sessions = rows["ga4_sessions_m"]
    codes = pd.DataFrame({
        "visibility_falling": (rows["impressions_m"] < VISIBILITY_DROP * rows["impressions_prev"]).isin([True]),
        "rank_unstable": rows["position_volatility_m"] >= UNSTABLE_SPREAD,
        "client_wide_low_ctr": client_factor(rows) < CLIENT_LOW,
        "commercial_intent": rows["main_intent"].isin(BUYING),
        "low_engagement": (sessions >= MIN_SESSIONS) & (rows["ga4_engaged_sessions_m"] < ENGAGED_SHARE * sessions),
    }, index=rows.index)
    rows = rows.join(codes)
    rows["action"] = np.select([codes[code] for code, _ in ACTION_ORDER], [action for _, action in ACTION_ORDER], DEFAULT_ACTION)
    rows["reason_codes"] = codes.apply(lambda row: ", ".join(["real_shortfall", *[code for code, applies in row.items() if applies]]), axis=1)
    rows["confidence"] = np.select([rows["probability"] >= HIGH, rows["probability"] >= MEDIUM], ["High", "Medium"], "Low")
    return rows


def queue_of(rows: pd.DataFrame) -> pd.DataFrame:
    """Return the month's queue (pages with a real shortfall, ranked by rule B) with the capped review list marked."""
    queue = rows[rows["rule_b"] > 0].sort_values("rule_b", ascending=False, kind="stable").reset_index(drop=True)
    queue["rank"] = queue.index + 1
    chosen = queue.index[queue.groupby("client_hash_id").cumcount() < CLIENT_CAP][:QUEUE_SIZE]
    queue["review_rank"] = pd.Series(range(1, len(chosen) + 1), index=chosen)
    return queue


backtest_model = fit_logistic(list(pairs))
may_rows = month_rows(BACKTEST_MONTH, backtest_model)
may_queue = queue_of(may_rows)

# What happened in June: the target where the page stayed eligible, otherwise why it dropped out.
june = outcome_pair.set_index("content_hash_id")
start, end = pd.Timestamp(f"{BACKTEST_MONTH}-01"), pd.Timestamp(f"{LIVE_MONTH}-01") + pd.offsets.MonthEnd(0)
in_pair = may_queue["content_hash_id"].isin(june.index)
ratio = (june["clicks_next"] / june["expected_bins_next"]).reindex(may_queue["content_hash_id"]).to_numpy()
seen_next = may_queue["content_hash_id"].isin(page_months.loc[page_months["month"] == LIVE_MONTH, "content_hash_id"])
may_queue["real_shortfall_next"] = may_queue["content_hash_id"].map(june["target"]).isin([True])
may_queue["missed_next"] = may_queue["content_hash_id"].map(june["missed_next"]).fillna(0)
may_queue["impressions_next"] = may_queue["content_hash_id"].map(june["impressions_next"])
# Lost visibility: June impressions under a fifth of May's, or no valid June row at all because the page vanished from search.
may_queue["lost_visibility"] = np.where(may_queue["impressions_next"].isna(), ~seen_next,
                                        may_queue["impressions_next"] < LOST_SHARE * may_queue["impressions_m"])
may_queue["outcome_next"] = np.select(
    [may_queue["real_shortfall_next"], in_pair & (ratio <= 0.5), in_pair,
     pd.to_datetime(may_queue["last_optimized_date"]).between(start, end), ~seen_next],
    ["Real shortfall again", "Short again, not beyond luck", "Recovered: over half of expected clicks",
     "Optimized by FlyRank during the pair", f"No impressions in {LIVE_MONTH}"],
    f"No valid position in {LIVE_MONTH}")
# Pages FlyRank optimized during the pair cannot be judged against June: someone already acted on them.
may_queue["optimized_during_pair"] = may_queue["outcome_next"] == "Optimized by FlyRank during the pair"
TOTAL_MISSED_NEXT = float(outcome_pair["missed_next"].sum())
show("Back-test queue", {
    "pages visible and scored in May": f"{len(may_rows):,}",
    "pages in the queue (real shortfall in May)": f"{len(may_queue):,}",
    "review list": f"{int(may_queue['review_rank'].notna().sum())} pages from {may_queue.loc[may_queue['review_rank'].notna(), 'client_hash_id'].nunique()} clients",
    "June's missed clicks on real shortfalls (all eligible pages)": f"{TOTAL_MISSED_NEXT:,.0f}",
})

### 1b. Choosing the queue size

The guide's threshold recipe: start with a simple rule, show what each cutoff captures, compare, and write down the trade-off accepted. A longer list catches more of next month's missed clicks but asks more of the editor and dilutes the hit rate; the client cap spreads the list across sites at some cost.

**Method:** this cell measures May's queue against June at several sizes: the share of picks that were real shortfalls again in June, the share of June's missed clicks they hold, how many clients they cover, and how many picks FlyRank optimized during the pair or that vanished from search. Pages FlyRank optimized during the pair were already acted on by someone else, so June cannot judge them; the hit rate is therefore also shown among the pages FlyRank left alone, the same exclusion the data contract makes. Pages that vanished from search still count as misses. It adds the chosen review list (50 pages, at most 10 per client) and the same 50 without the cap.


In [ ]:
SIZES = [20, 50, 100, 200, 500]


def size_row(picks: pd.DataFrame, label: str) -> dict[str, object]:
    """Return what one set of picks captured in June, overall and among the pages FlyRank left alone."""
    alone = picks[~picks["optimized_during_pair"]]
    return {"List": label, "Pages": len(picks), "Optimized by FlyRank": int(picks["optimized_during_pair"].sum()),
            "Vanished from search": int(picks["outcome_next"].isin([f"No impressions in {LIVE_MONTH}", f"No valid position in {LIVE_MONTH}"]).sum()),
            "Real shortfall again, all picks": picks["real_shortfall_next"].mean(),
            "Real shortfall again, pages FlyRank left alone": alone["real_shortfall_next"].mean() if len(alone) else np.nan,
            "Share of June's missed clicks": picks["missed_next"].sum() / TOTAL_MISSED_NEXT, "Clients": picks["client_hash_id"].nunique()}


tradeoff = pd.DataFrame([size_row(may_queue.head(size), f"Top {size}") for size in SIZES if size <= len(may_queue)]
                        + [size_row(may_queue[may_queue["review_rank"].notna()], f"Top {QUEUE_SIZE}, at most {CLIENT_CAP} per client (chosen)")])
show_table("Queue size against what it captured in June (May back-test)", tradeoff.assign(
    **{column: tradeoff[column].map("{:.0%}".format) for column in ["Real shortfall again, all picks", "Real shortfall again, pages FlyRank left alone"]},
    **{"Share of June's missed clicks": tradeoff["Share of June's missed clicks"].map("{:.1%}".format)}).set_index("List"))

**Method:** this cell draws the trade-off in two panels on the same queue sizes: the hit rate among the pages FlyRank left alone on the left and the share of June's missed clicks on the right. The uncapped queue is the blue line; the chosen review list is the orange point. The title states the finding; the table above holds the values.


In [ ]:
def load_inter() -> str:
    """Register the Inter font for this session and return the font family to draw with.

    The 400 and 600 weights are cached in the system temp folder, never in the project. If the
    download is unavailable, the chart falls back to matplotlib's default font, so the notebook still runs offline.
    """
    cache = os.path.join(tempfile.gettempdir(), "inter-font")
    os.makedirs(cache, exist_ok=True)
    try:
        for weight in (400, 600):
            path = os.path.join(cache, f"inter-{weight}.ttf")
            if not os.path.exists(path):
                urllib.request.urlretrieve(f"https://cdn.jsdelivr.net/fontsource/fonts/inter@latest/latin-{weight}-normal.ttf", path)
            font_manager.fontManager.addfont(path)
    except OSError:
        return "DejaVu Sans"
    return "Inter"


INK, SECONDARY, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#ecebe7", "#fcfcfb"
UNCAPPED, CHOSEN = "#2a78d6", "#eb6834"
plt.rcParams.update({"font.family": load_inter()})
curve, chosen_row = tradeoff[tradeoff["List"].str.startswith("Top ") & ~tradeoff["List"].str.contains("chosen")], tradeoff.iloc[-1]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), dpi=120, facecolor=SURFACE)
fig.subplots_adjust(left=0.07, right=0.97, top=0.72, bottom=0.15, wspace=0.28)
for ax, column, form, heading in ((axes[0], "Real shortfall again, pages FlyRank left alone", "{:.0%}", "Picks FlyRank left alone that were short again in June"),
                                  (axes[1], "Share of June's missed clicks", "{:.0%}", "Share of June's missed clicks the list holds")):
    ax.set_facecolor(SURFACE)
    ax.plot(curve["Pages"], curve[column], color=UNCAPPED, linewidth=2, marker="o", markersize=8, markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=2)
    ax.plot(chosen_row["Pages"], chosen_row[column], "o", markersize=11, color=CHOSEN, markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)
    for pages, value in zip(curve["Pages"], curve[column], strict=True):
        ax.annotate(form.format(value), (pages, value), xytext=(0, 9), textcoords="offset points", ha="center", color=SECONDARY, fontsize=9)
    ax.set_xscale("log")
    ax.set_xticks(curve["Pages"], [str(pages) for pages in curve["Pages"]])
    ax.minorticks_off()
    top = max(curve[column].max(), chosen_row[column])
    ax.set_ylim(0, min(top * 1.25, 1.12))
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=6))
    ax.set_yticks([tick for tick in ax.get_yticks() if 0 <= tick <= 1.0])  # a share never passes 100%
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_xlabel("Pages in the list", color=MUTED, fontsize=10, labelpad=6)
    ax.set_title(heading, color=INK, fontsize=11, loc="left", pad=8)
    ax.grid(axis="y", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(length=0, colors=MUTED, labelsize=9.5, pad=5)
    for spine in ax.spines.values():
        spine.set_visible(False)
hits_at = dict(zip(curve["Pages"], curve["Real shortfall again, pages FlyRank left alone"], strict=True))
clicks_at = dict(zip(curve["Pages"], curve["Share of June's missed clicks"], strict=True))
longest = curve["Pages"].max()
hits_change = hits_at[longest] - hits_at[QUEUE_SIZE]
hits_words = (f"hits fall from {hits_at[QUEUE_SIZE]:.0%} to {hits_at[longest]:.0%}" if hits_change < -0.005 else
              f"hits rise from {hits_at[QUEUE_SIZE]:.0%} to {hits_at[longest]:.0%}" if hits_change > 0.005 else f"hits stay near {hits_at[QUEUE_SIZE]:.0%}")
title = (f"Growing the list from {QUEUE_SIZE} to {longest} pages raises the clicks it holds from "
         f"{clicks_at[QUEUE_SIZE]:.0%} to {clicks_at[longest]:.0%}, while {hits_words}")
fig.text(0.5, 0.93, title, ha="center", color=INK, fontsize=13, fontweight="semibold")
key = [Line2D([], [], color=UNCAPPED, linewidth=2, marker="o", markersize=8, markeredgecolor=SURFACE, label="queue in rule B order, no cap"),
       Line2D([], [], color=CHOSEN, linestyle="none", marker="o", markersize=10, markeredgecolor=SURFACE, label=f"chosen review list: {QUEUE_SIZE} pages, at most {CLIENT_CAP} per client")]
fig.legend(handles=key, loc="upper center", bbox_to_anchor=(0.5, 0.87), ncol=2, frameon=False, fontsize=10, labelcolor=SECONDARY, columnspacing=2)
plt.show()

### 1c. Does the confidence label mean anything?

The baseline notebook's confidence note did not separate hits from misses. The label here comes from a model with roughly honest probabilities, so it can be checked directly: pages marked high should turn out to be real shortfalls again more often than pages marked low, and close to the probability the model gave them.

**Method:** this cell groups every page in May's queue, and separately the review list, by confidence label, and compares the model's average probability with the share that were real shortfalls again in June, over all pages and over the pages FlyRank left alone (the model learned on pages the data contract keeps, which excludes pages optimized during the pair). It then shows the June outcome by action.


In [ ]:
BANDS = ["High", "Medium", "Low"]
calibration = []
for scope, picks in (("Whole queue", may_queue), ("Review list", may_queue[may_queue["review_rank"].notna()])):
    for band in BANDS:
        inside = picks[picks["confidence"] == band]
        alone = inside[~inside["optimized_during_pair"]]
        calibration.append({"Scope": scope, "Confidence": band, "Pages": len(inside),
                            "Model's average probability": f"{inside['probability'].mean():.0%}" if len(inside) else "n/a",
                            "Real shortfall again, all pages": f"{inside['real_shortfall_next'].mean():.0%}" if len(inside) else "n/a",
                            "Pages FlyRank left alone": len(alone),
                            "Real shortfall again, pages FlyRank left alone": f"{alone['real_shortfall_next'].mean():.0%}" if len(alone) else "n/a"})
show_table("Confidence label against the June outcome (May back-test)", pd.DataFrame(calibration).set_index(["Scope", "Confidence"]))

by_code = []
for action in [*(action for _, action in ACTION_ORDER), DEFAULT_ACTION]:
    inside = may_queue[may_queue["action"] == action]
    alone = inside[~inside["optimized_during_pair"]]
    by_code.append({"Action": action, "Pages in the queue": len(inside), "Optimized by FlyRank": int(inside["optimized_during_pair"].sum()),
                    "Real shortfall again, pages FlyRank left alone": f"{alone['real_shortfall_next'].mean():.0%}" if len(alone) else "n/a",
                    "Lost visibility in June": f"{inside['lost_visibility'].mean():.0%}" if len(inside) else "n/a"})
show_table("June outcome by action (May back-test, whole queue)", pd.DataFrame(by_code).set_index("Action"))

### 1d. Where the playbook disagreed or was wrong

The guide asks for 10 to 15 rows where the model disagreed with the baseline or with reality, read one by one. Twelve rows are chosen by fixed rules, four of each kind, so every kind of disagreement is seen:

- **(a) Rule B ranks it in its top 50, the model gives it low confidence.**
- **(b) The model is highly confident, but the page is not in the queue,** because it was not yet short in May and rule B cannot see it.
- **(c) High-confidence queue picks that were not real shortfalls in June.**

Kinds (b) and (c) leave out pages FlyRank optimized during May or June: someone already acted on them, so June cannot say whether the playbook was right.

**Method:** this cell picks the twelve rows from the May back-test and shows each one's May numbers, both views, and what happened in June.


In [ ]:
# Pages FlyRank optimized during the pair are left out of kinds (b) and (c): someone already acted on them, so June cannot judge them.
never_queued = may_rows[(may_rows["rule_b"] == 0) & ~pd.to_datetime(may_rows["last_optimized_date"]).between(start, end)]
never_queued = never_queued.sort_values("probability", ascending=False)
never_ratio = never_queued["content_hash_id"].map(june["clicks_next"] / june["expected_bins_next"])
never_in_pair = never_queued["content_hash_id"].isin(june.index)
never_seen = never_queued["content_hash_id"].isin(page_months.loc[page_months["month"] == LIVE_MONTH, "content_hash_id"])
never_queued = never_queued.assign(
    real_shortfall_next=never_queued["content_hash_id"].map(june["target"]).isin([True]), ratio_next=never_ratio,
    impressions_next=never_queued["content_hash_id"].map(june["impressions_next"]),
    outcome_next=np.select([never_queued["content_hash_id"].map(june["target"]).isin([True]), never_in_pair & (never_ratio <= 0.5), never_in_pair, ~never_seen],
                           ["Real shortfall", "Short, not beyond luck", "Clicking normally", f"No impressions in {LIVE_MONTH}"], f"No valid position in {LIVE_MONTH}"))
kinds = {
    "(a) Rule B top 50, model low": may_queue[(may_queue["rank"] <= QUEUE_SIZE) & (may_queue["confidence"] == "Low")].head(4),
    "(b) Model high, not in the queue": never_queued[never_queued["confidence"] == "High"].head(4),
    "(c) High confidence, not short in June": may_queue[(may_queue["confidence"] == "High") & ~may_queue["real_shortfall_next"]
                                                         & ~may_queue["optimized_during_pair"]].head(4),
}
error_rows = []
for kind, picks in kinds.items():
    if picks.empty:
        error_rows.append({"Kind": kind, "Rule B rank": "none found"})
    for _, row in picks.iterrows():
        june_ratio = row["ratio_next"] if "ratio_next" in row.index else june["clicks_next"].get(row["content_hash_id"], np.nan) / june["expected_bins_next"].get(row["content_hash_id"], np.nan)
        error_rows.append({"Kind": kind, "Rule B rank": f"{row['rank']:,.0f}" if pd.notna(row.get("rank", np.nan)) else "not queued",
                           "Probability": f"{row['probability']:.0%}", "May impressions": f"{row['impressions_m']:,.0f}",
                           "May clicks": f"{row['clicks_m']:,.0f}", "May expected clicks": f"{row['expected_bins_m']:,.1f}",
                           "Position": f"{row['position_m']:.1f}", "Position spread": f"{row['position_volatility_m']:.1f}",
                           "Search intent": row["main_intent"] if pd.notna(row["main_intent"]) else "unknown",
                           "Action": row["action"], "June impressions": f"{row['impressions_next']:,.0f}" if pd.notna(row["impressions_next"]) else "none",
                           "June clicks ÷ expected": f"{june_ratio:.2f}" if pd.notna(june_ratio) else "n/a",
                           "June outcome": row["outcome_next"]})
show_table("Twelve rows where the playbook disagreed with itself or with June", pd.DataFrame(error_rows).set_index("Kind").fillna(""))

### 1e. The live queue: June 2026, for July

**Method:** this cell scores every page visible in June 2026 with the model trained on all four labelled pairs, builds June's queue, and shows the review list: rank, action, reason codes, confidence, and the numbers behind each pick. The data is pseudonymized, so pages are identified by their position in the list; the identifiers go only to the local queue file written in section 5.


In [ ]:
live_model = fit_logistic(list(labelled))
june_rows = month_rows(LIVE_MONTH, live_model)
june_queue = queue_of(june_rows)
review = june_queue[june_queue["review_rank"].notna()].sort_values("review_rank")
show("Live queue", {"pages visible and scored in June": f"{len(june_rows):,}", "pages in the queue (real shortfall in June)": f"{len(june_queue):,}",
                    "review list": f"{len(review)} pages from {review['client_hash_id'].nunique()} clients"})
show_table(f"Review list for July (June 2026 data, {QUEUE_SIZE} pages, at most {CLIENT_CAP} per client)", pd.DataFrame({
    "Action": review["action"].to_numpy(), "Reason codes": review["reason_codes"].to_numpy(), "Confidence": review["confidence"].to_numpy(),
    "Probability": review["probability"].map("{:.0%}".format).to_numpy(), "Missed clicks in June": review["rule_b"].map("{:,.0f}".format).to_numpy(),
    "Impressions": review["impressions_m"].map("{:,.0f}".format).to_numpy(), "Position": review["position_m"].map("{:.1f}".format).to_numpy(),
    "Position spread": review["position_volatility_m"].map("{:.1f}".format).to_numpy(),
    "Search intent": review["main_intent"].fillna("unknown").to_numpy(),
}, index=pd.Index(review["review_rank"].astype(int), name="Review rank")))
show_table("Actions in the review list and in the whole queue", pd.DataFrame({
    "Review list": review["action"].value_counts(), "Whole queue": june_queue["action"].value_counts()}).fillna(0).astype(int).rename_axis("Action"))

## 2. Intended use and limits

**Who uses it, for what.** A content editor or SEO lead at FlyRank, once a month, to decide which existing pages to look at first for lost click-through. It is a prioritized reading list with a suggested first step for each page, not a list of guaranteed wins.

**Where it is valid.**

- Pages with a valid position, on clients whose search history covers the whole month.
- Click-through problems relative to the page's own position and site: the expectation is client-aware.
- The month after the one it was built from. The evidence is that real shortfalls repeat in the next month (3.8 to 4.8 times as often in the validation audit), not that they last indefinitely.

**Where it stops being valid.**

- **Small pages.** A real shortfall needs enough traffic to rule out luck, so pages with a few hundred impressions rarely qualify; the queue says little about them.
- **New clients and new pages.** Clients whose history starts mid-month are left out; pages with no previous month get no visibility check.
- **Visibility problems.** A page dropping out of search is flagged only when its impressions already halved; the queue is not built to predict de-indexing.
- **Causes and gains.** The queue shows where clicks are missing, not why, and not how many a fix would bring back. No experiment has measured that.
- **Seasonal swings and big search-engine changes.** The model learned on four months (February to June 2026); a different season or a results-page redesign can move the baseline.

**Method:** this cell measures the live queue's coverage: how many June pages and impressions are scored, queued, and reviewed, and how the queue spreads across page sizes.


In [ ]:
visible_june = page_months[page_months["month"] == LIVE_MONTH]
coverage = pd.DataFrame({
    "Pages": [len(visible_june), len(june_rows), len(june_queue), len(review)],
    "Impressions": [visible_june["impressions"].sum(), june_rows["impressions_m"].sum(), june_queue["impressions_m"].sum(), review["impressions_m"].sum()],
}, index=pd.Index(["All pages with June impressions", "Scored (valid position, covered client)", "In the queue (real shortfall)", "In the review list"], name="Stage"))
show_table("Coverage of the live queue, June 2026", coverage.assign(
    **{"Share of June impressions": (coverage["Impressions"] / coverage["Impressions"].iloc[0]).map("{:.1%}".format)},
    Pages=coverage["Pages"].map("{:,}".format), Impressions=coverage["Impressions"].map("{:,.0f}".format)))
SIZE_BANDS = {"Under 100": (0, 99), "100 to 999": (100, 999), "1,000 to 9,999": (1_000, 9_999), "10,000 or more": (10_000, np.inf)}
show_table("Who the queue reaches, by page size (June 2026)", pd.DataFrame({
    band: {"Scored pages": f"{june_rows['impressions_m'].between(low, high).sum():,}",
           "Share in the queue": f"{june_rows.loc[june_rows['impressions_m'].between(low, high), 'rule_b'].gt(0).mean():.1%}",
           "In the review list": int(review["impressions_m"].between(low, high).sum())}
    for band, (low, high) in SIZE_BANDS.items()}).T.rename_axis("Impressions in June"))

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.